In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import os
from supabase import create_client

plt.style.use('seaborn-v0_8-darkgrid')

In [2]:
def load_es_15min_with_progress(start_date, end_date):
    """
    Load ES 15-minute data from Supabase.
    """
    supabase = create_client(
        os.getenv("SUPABASE_URL"),
        os.getenv("SUPABASE_SERVICE_ROLE_KEY")
    )
    count_res = supabase.table('es_15min').select('*', count='exact').gte('timestamp', start_date).lte('timestamp', end_date).limit(0).execute()
    total = count_res.count
    print(f"📊 Total rows in period: {total}")
    all_data = []
    offset = 0
    batch = 1000
    with tqdm(total=total, desc="Downloading data", unit="rows") as pbar:
        while True:
            query = supabase.table('es_15min').select('*').order('timestamp').range(offset, offset+batch-1)
            query = query.gte('timestamp', start_date).lte('timestamp', end_date)
            response = query.execute()
            if not response.data:
                break
            all_data.extend(response.data)
            pbar.update(len(response.data))
            offset += batch
    df = pd.DataFrame(all_data)
    time_col = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
    if time_col:
        df = df.rename(columns={time_col[0]: 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df.columns = [c.lower() for c in df.columns]
    return df

In [3]:
# Load 2+ years of data
df_all = load_es_15min_with_progress("2023-01-01", "2026-05-25")
print(f"✅ Loaded {len(df_all)} candles")
print(f"Range: {df_all['timestamp'].min()} → {df_all['timestamp'].max()}")

# %%
# Filter only Tuesdays (weekday = 1) and market hours (8:30 - 16:00 ET)
df_all['weekday'] = df_all['timestamp'].dt.weekday
df_all['hour_min'] = df_all['timestamp'].dt.hour + df_all['timestamp'].dt.minute/60
df_tuesdays = df_all[(df_all['weekday'] == 1) & (df_all['hour_min'] >= 8.5) & (df_all['hour_min'] <= 16.0)].copy()
print(f"Tuesdays in market hours: {len(df_tuesdays)} candles")
print(f"Unique Tuesday dates: {df_tuesdays['timestamp'].dt.date.nunique()}")


📊 Total rows in period: 77736


✅ Loaded 77736 candles
Range: 2023-01-02 18:00:00 → 2026-05-22 20:45:00
Tuesdays in market hours: 5252 candles
Unique Tuesday dates: 170


In [4]:
BB_PERIOD = 20
BB_STD = 1.2
RSI_PERIOD = 14
RSI_OVERSOLD = 30
RSI_OVERBOUGHT = 60
ATR_PERIOD = 14
STOP_MULT = 0.8
TAKE_MULT = 1.5
COMMISSION = 1.20   # per round-trip for MES
INITIAL_CAPITAL = 25000

In [5]:
def add_indicators(df):
    """
    Add Bollinger Bands, RSI, and ATR to a DataFrame.
    """
    df = df.copy()
    # Bollinger Bands
    df['bb_mid'] = df['close'].rolling(BB_PERIOD).mean()
    df['bb_std'] = df['close'].rolling(BB_PERIOD).std()
    df['bb_upper'] = df['bb_mid'] + BB_STD * df['bb_std']
    df['bb_lower'] = df['bb_mid'] - BB_STD * df['bb_std']
    # RSI
    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(RSI_PERIOD).mean()
    avg_loss = loss.rolling(RSI_PERIOD).mean()
    rs = avg_gain / avg_loss
    df['rsi'] = 100 - (100 / (1 + rs))
    # ATR
    high_low = df['high'] - df['low']
    high_close = (df['high'] - df['close'].shift()).abs()
    low_close = (df['low'] - df['close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df['atr'] = tr.rolling(ATR_PERIOD).mean()
    return df

In [6]:
def backtest_mean_reversion(dates_list, df_market, stop_mult=STOP_MULT, take_mult=TAKE_MULT,
                            rsi_buy=RSI_OVERSOLD, rsi_sell=RSI_OVERBOUGHT, bb_std=BB_STD):
    """
    Backtest the mean reversion strategy on a list of dates.
    Returns a DataFrame with trades (pnl_usd, direction, date).
    """
    trades = []
    for date in tqdm(dates_list, desc="Backtesting"):
        day = df_market[df_market['timestamp'].dt.date == date].copy()
        if len(day) < 20:
            continue
        day = add_indicators(day)
        # Override BB std if different from default
        day['bb_upper'] = day['bb_mid'] + bb_std * day['bb_std']
        day['bb_lower'] = day['bb_mid'] - bb_std * day['bb_std']
        
        position = None
        entry_price = 0
        entry_time = None
        stop_price = 0
        target_price = 0
        
        for i, row in day.iterrows():
            # Time stop: close at 15:45 if still in position
            if position is not None and (row['timestamp'].hour == 15 and row['timestamp'].minute >= 45):
                exit_price = row['close']
                pnl_pts = (exit_price - entry_price) if position == 1 else (entry_price - exit_price)
                pnl_usd = pnl_pts * 5 - COMMISSION
                trades.append({'date': date, 'pnl_usd': pnl_usd, 'direction': position})
                position = None
                continue
            
            # Manage open position
            if position is not None:
                if position == 1:  # long
                    if row['high'] >= target_price:
                        pnl_pts = take_mult * (day.loc[entry_time, 'atr'] if entry_time in day.index else row['atr'])
                        pnl_usd = pnl_pts * 5 - COMMISSION
                        trades.append({'date': date, 'pnl_usd': pnl_usd, 'direction': 1})
                        position = None
                    elif row['low'] <= stop_price:
                        pnl_pts = -stop_mult * (day.loc[entry_time, 'atr'] if entry_time in day.index else row['atr'])
                        pnl_usd = pnl_pts * 5 - COMMISSION
                        trades.append({'date': date, 'pnl_usd': pnl_usd, 'direction': 1})
                        position = None
                else:  # short
                    if row['low'] <= target_price:
                        pnl_pts = take_mult * (day.loc[entry_time, 'atr'] if entry_time in day.index else row['atr'])
                        pnl_usd = pnl_pts * 5 - COMMISSION
                        trades.append({'date': date, 'pnl_usd': pnl_usd, 'direction': -1})
                        position = None
                    elif row['high'] >= stop_price:
                        pnl_pts = -stop_mult * (day.loc[entry_time, 'atr'] if entry_time in day.index else row['atr'])
                        pnl_usd = pnl_pts * 5 - COMMISSION
                        trades.append({'date': date, 'pnl_usd': pnl_usd, 'direction': -1})
                        position = None
            
            # New signal (only if flat)
            if position is None and not pd.isna(row['rsi']):
                # Long signal: close below lower BB and RSI < oversold
                if row['close'] < row['bb_lower'] and row['rsi'] < rsi_buy:
                    position = 1
                    entry_price = row['close']
                    entry_time = row['timestamp']
                    atr_val = row['atr']
                    stop_price = entry_price - stop_mult * atr_val
                    target_price = entry_price + take_mult * atr_val
                # Short signal: close above upper BB and RSI > overbought
                elif row['close'] > row['bb_upper'] and row['rsi'] > rsi_sell:
                    position = -1
                    entry_price = row['close']
                    entry_time = row['timestamp']
                    atr_val = row['atr']
                    stop_price = entry_price + stop_mult * atr_val
                    target_price = entry_price - take_mult * atr_val
    return pd.DataFrame(trades)


In [7]:
unique_tuesdays = sorted(df_tuesdays['timestamp'].dt.date.unique())
print(f"Total Tuesdays in dataset: {len(unique_tuesdays)}")

trades_all = backtest_mean_reversion(unique_tuesdays, df_tuesdays,
                                     stop_mult=STOP_MULT, take_mult=TAKE_MULT,
                                     rsi_buy=RSI_OVERSOLD, rsi_sell=RSI_OVERBOUGHT,
                                     bb_std=BB_STD)

print(f"\nTotal trades generated: {len(trades_all)}")
print(trades_all.head())


Total Tuesdays in dataset: 170


Backtesting: 100%|██████████| 170/170 [00:00<00:00, 288.37it/s]


Total trades generated: 245
         date    pnl_usd  direction
0  2023-01-10 -36.200000         -1
1  2023-01-24  45.407143         -1
2  2023-01-31 -20.128571         -1
3  2023-01-31  35.764286         -1
4  2023-01-31 -81.200000         -1


In [8]:
def compute_metrics(trades_df, capital=INITIAL_CAPITAL):
    if len(trades_df) == 0:
        return {}
    pnls = trades_df['pnl_usd']
    win_rate = (pnls > 0).mean()
    gross_profit = pnls[pnls>0].sum()
    gross_loss = abs(pnls[pnls<0].sum())
    profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.inf
    net_profit = pnls.sum()
    equity = capital + pnls.cumsum()
    max_dd = (equity - equity.cummax()).min()
    returns = pnls / capital
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() != 0 else 0
    t_stat, p_val = stats.ttest_1samp(returns, 0)
    return {
        'total_trades': len(trades_df),
        'win_rate': win_rate,
        'profit_factor': profit_factor,
        'net_profit': net_profit,
        'sharpe_ratio': sharpe,
        'max_drawdown_pct': max_dd / capital,
        't_stat': t_stat,
        'p_value': p_val
    }

metrics = compute_metrics(trades_all)
print("\n=== FINAL BACKTEST METRICS (ALL TUESDAYS) ===")
for k, v in metrics.items():
    if isinstance(v, float):
        if k == 'net_profit':
            print(f"{k}: ${v:,.2f}")
        elif k == 'max_drawdown_pct':
            print(f"{k}: {v:.2%}")
        else:
            print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")



=== FINAL BACKTEST METRICS (ALL TUESDAYS) ===
total_trades: 245
win_rate: 0.3714
profit_factor: 1.0113
net_profit: $56.31
sharpe_ratio: 0.0606
max_drawdown_pct: -2.83%
t_stat: 0.0597
p_value: 0.9524


In [9]:
# Trade frequency analysis
tuesdays_with_trade = set(trades_all['date'].unique())
print(f"\n=== TRADE FREQUENCY ===")
print(f"Tuesdays with at least one trade: {len(tuesdays_with_trade)} / {len(unique_tuesdays)} ({len(tuesdays_with_trade)/len(unique_tuesdays):.1%})")
print(f"Probability of having a trade next Tuesday: {len(tuesdays_with_trade)/len(unique_tuesdays):.1%}")

# Distribution of trades per Tuesday
trades_per_day = trades_all.groupby('date').size().value_counts().sort_index()
print("\nTrades per Tuesday distribution:")
for n, count in trades_per_day.items():
    print(f"  {n} trade(s): {count} Tuesdays ({count/len(unique_tuesdays):.1%})")



=== TRADE FREQUENCY ===
Tuesdays with at least one trade: 127 / 170 (74.7%)
Probability of having a trade next Tuesday: 74.7%

Trades per Tuesday distribution:
  1 trade(s): 61 Tuesdays (35.9%)
  2 trade(s): 31 Tuesdays (18.2%)
  3 trade(s): 23 Tuesdays (13.5%)
  4 trade(s): 8 Tuesdays (4.7%)
  5 trade(s): 3 Tuesdays (1.8%)
  6 trade(s): 1 Tuesdays (0.6%)


In [10]:
def monte_carlo_simulation(trades_df, n_sim=5000):
    pnls = trades_df['pnl_usd'].values.copy()
    if len(pnls) == 0:
        return []
    final_capitals = []
    for _ in range(n_sim):
        shuffled = np.random.permutation(pnls)
        equity = INITIAL_CAPITAL + np.cumsum(shuffled)
        final_capitals.append(equity[-1])
    return final_capitals

final_eqs = monte_carlo_simulation(trades_all, n_sim=5000)
if final_eqs:
    print("\n=== MONTE CARLO (5000 simulations) ===")
    print(f"Mean final capital: ${np.mean(final_eqs):,.0f}")
    print(f"5th percentile: ${np.percentile(final_eqs, 5):,.0f}")
    print(f"95th percentile: ${np.percentile(final_eqs, 95):,.0f}")
    prob_loss = np.mean(np.array(final_eqs) < INITIAL_CAPITAL)
    print(f"Probability of loss: {prob_loss:.2%}")
else:
    print("No trades to run Monte Carlo.")


=== MONTE CARLO (5000 simulations) ===
Mean final capital: $25,056
5th percentile: $25,056
95th percentile: $25,056
Probability of loss: 0.00%
